# Fase 3 · Representación VSM con TF-IDF (scikit-learn)

Este notebook construye las representaciones **TF-IDF** del corpus normalizado (`data/scientific_normalized_corpus.tsv`) con `TfidfVectorizer` de scikit-learn.

**Objetivo:** generar las 4 configuraciones TF-IDF del plan, {título, abstract} × {unigrama, bigrama}, y guardar cada vectorizador con su matriz en `models/{campo}_{ngrama}_tfidf.pkl`. Las configuraciones de frecuencia y binaria se agregan después con la misma estructura.

## Decisiones de diseño

| Decisión | Valor | Motivo |
|---|---|---|
| Texto de entrada | `title_norm` y `abstract_norm` | Los vectorizadores se ajustan sobre el texto ya normalizado (sección 6 del plan) |
| `lowercase` | `False` | La normalización ya pasó todo a minúsculas |
| `token_pattern` | `r"\S+"` (separar por espacios) | El texto normalizado ya viene tokenizado por spaCy y unido con espacios. El patrón por defecto (`(?u)\b\w\w+\b`) descartaría tokens de un carácter (números como `3`, variables como `x`) y volvería a partir tokens con símbolos |
| `stop_words` | `None` | Las stopwords ya se quitaron por POS en la normalización |
| `ngram_range` | `(1, 1)` unigramas, `(2, 2)` bigramas | Lo pide el plan. Los bigramas se forman **después** de quitar stopwords: *analysis of text* → `analysis text` |
| `min_df` / `max_df` | Por defecto (`1` y `1.0`) | Con ~500 documentos no se poda vocabulario; así ninguna consulta pierde términos raros |
| `norm` | `"l2"` (por defecto) | Cada fila queda con norma 1, así que la similitud coseno es un producto punto |
| `smooth_idf`, `sublinear_tf` | `True`, `False` (por defecto) | idf = ln((1 + n) / (1 + df)) + 1; tf = conteo crudo |
| Formato guardado | `pickle` con vectorizador, matriz CSR y `doc_ids` | `doc_ids` (= `arxiv_id`) relaciona cada fila de la matriz con su documento en la Fase 4 |

## 1. Configuración

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

NORM_PATH = ROOT / "data" / "scientific_normalized_corpus.tsv"
MODELS_DIR = ROOT / "models"
REPORT_PATH = ROOT / "results" / "tfidf_stats.md"
MODELS_DIR.mkdir(exist_ok=True)

# Campo del CLI -> columna normalizada del TSV
FIELDS = {"title": "title_norm", "abstract": "abstract_norm"}
# Tipo de n-grama del CLI -> ngram_range de scikit-learn
NGRAMS = {"unigram": (1, 1), "bigram": (2, 2)}
REPR = "tfidf"

print("scikit-learn", sklearn.__version__)

## 2. Carga del corpus normalizado

In [ ]:
corpus = pd.read_csv(NORM_PATH, sep="\t", dtype=str, keep_default_na=False)
assert corpus["arxiv_id"].is_unique
doc_ids = corpus["arxiv_id"].tolist()

for col in FIELDS.values():
    empty = (corpus[col].str.strip() == "").sum()
    print(f"{col}: {len(corpus)} documentos, {empty} vacíos")

## 3. Vectorizador TF-IDF

`build_vectorizer` es la única fábrica de vectorizadores, para que las 4 configuraciones usen exactamente los mismos parámetros salvo `ngram_range`.

In [ ]:
def build_vectorizer(ngram_range) -> TfidfVectorizer:
    return TfidfVectorizer(
        lowercase=False,       # el texto ya está en minúsculas
        token_pattern=r"\S+",  # los tokens ya vienen separados por espacios
        ngram_range=ngram_range,
        norm="l2",
        smooth_idf=True,
        sublinear_tf=False,
    )


def model_path(field: str, ngram: str, repr_: str = REPR) -> Path:
    return MODELS_DIR / f"{field}_{ngram}_{repr_}.pkl"

## 4. Ajuste y guardado de las 4 configuraciones

In [ ]:
rows = []
for field, col in FIELDS.items():
    for ngram, ngram_range in NGRAMS.items():
        vectorizer = build_vectorizer(ngram_range)
        matrix = vectorizer.fit_transform(corpus[col]).tocsr()

        payload = {
            "field": field,
            "ngram": ngram,
            "repr": REPR,
            "column": col,
            "vectorizer": vectorizer,
            "matrix": matrix,
            "doc_ids": doc_ids,
        }
        path = model_path(field, ngram)
        with open(path, "wb") as f:
            pickle.dump(payload, f)

        n_docs, n_terms = matrix.shape
        empty_rows = int((matrix.getnnz(axis=1) == 0).sum())
        rows.append({
            "config": path.stem,
            "documentos": n_docs,
            "vocabulario": n_terms,
            "no_ceros": matrix.nnz,
            "densidad_%": round(100 * matrix.nnz / (n_docs * n_terms), 3),
            "dispersion_%": round(100 * (1 - matrix.nnz / (n_docs * n_terms)), 3),
            "terminos_por_doc": round(matrix.nnz / n_docs, 1),
            "docs_sin_terminos": empty_rows,
        })
        print("Guardado", path.relative_to(ROOT))

stats = pd.DataFrame(rows).set_index("config")
stats

## 5. Verificaciones

1. Cada `.pkl` se recarga y reproduce la misma matriz.
2. Las filas tienen norma L2 = 1 (o 0 si el documento quedó vacío).
3. Transformar de nuevo el corpus con el vectorizador guardado da la misma matriz (es lo que se hará con las consultas en la Fase 4).

In [ ]:
for field, col in FIELDS.items():
    for ngram in NGRAMS:
        with open(model_path(field, ngram), "rb") as f:
            m = pickle.load(f)
        X = m["matrix"]
        assert m["doc_ids"] == doc_ids
        assert X.shape == (len(corpus), len(m["vectorizer"].vocabulary_))

        norms = np.sqrt(X.multiply(X).sum(axis=1)).A1
        assert np.allclose(norms[X.getnnz(axis=1) > 0], 1.0)

        X2 = m["vectorizer"].transform(corpus[col])
        assert abs(X - X2).max() < 1e-12
print("Verificaciones correctas")

## 6. Ejemplos

Términos con mayor peso TF-IDF de un documento, y una consulta de prueba ya normalizada a mano (en la Fase 4 se pasará antes por `normalize()`). Como las filas tienen norma 1, la similitud coseno es `X @ q.T`.

In [ ]:
def load_model(field: str, ngram: str, repr_: str = REPR) -> dict:
    with open(model_path(field, ngram, repr_), "rb") as f:
        return pickle.load(f)


def top_terms(model: dict, row: int, k: int = 10) -> pd.DataFrame:
    vec = model["matrix"][row]
    terms = model["vectorizer"].get_feature_names_out()
    order = np.argsort(-vec.data)[:k]
    return pd.DataFrame({"termino": terms[vec.indices[order]], "tfidf": vec.data[order].round(4)})


m = load_model("abstract", "unigram")
print(corpus.loc[0, "arxiv_id"], "|", corpus.loc[0, "title"])
top_terms(m, 0)

In [ ]:
query_norm = "misinformation detection use transformer"  # ya normalizada a mano
q = m["vectorizer"].transform([query_norm])
print("Términos de la consulta en el vocabulario:", q.nnz)

scores = (m["matrix"] @ q.T).toarray().ravel()
top = pd.DataFrame({"arxiv_id": doc_ids, "score": scores, "title": corpus["title"]})
top = top.sort_values(["score", "arxiv_id"], ascending=[False, True], kind="stable").head(10)
top.insert(0, "rank", range(1, 11))
top.reset_index(drop=True)

## 7. Reporte de la representación TF-IDF

Se guarda en `results/tfidf_stats.md`.

In [ ]:
from datetime import datetime, timezone

report = f"""# Representación TF-IDF

- Generado: {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}
- scikit-learn {sklearn.__version__}, `TfidfVectorizer`
- Parámetros: `lowercase=False`, `token_pattern=r"\\S+"`, `norm="l2"`, `smooth_idf=True`, `sublinear_tf=False`, sin `min_df`/`max_df`
- Entrada: `{NORM_PATH.relative_to(ROOT)}` ({len(corpus)} documentos)
- Modelos: `models/{{campo}}_{{ngrama}}_tfidf.pkl`

## Vocabulario y dispersión
{stats.to_markdown()}
"""
REPORT_PATH.write_text(report, encoding="utf-8")
print(REPORT_PATH)

## 8. Resultado

Archivos generados:
- `models/title_unigram_tfidf.pkl`, `models/title_bigram_tfidf.pkl`, `models/abstract_unigram_tfidf.pkl`, `models/abstract_bigram_tfidf.pkl`: cada uno con `vectorizer`, `matrix` (CSR, filas con norma L2 = 1) y `doc_ids`.
- `results/tfidf_stats.md`: tamaño de vocabulario y dispersión de cada matriz.

**Siguiente paso:** configuraciones de frecuencia y binaria (`CountVectorizer`) y Fase 4, recuperación con similitud coseno.